# SpaceLLaVA: FP16 vs NF4 control v12

## Goal

Проверить влияние NF4: **те же 16 запросов в FP16 и NF4 на одной физической GPU**, один checkpoint, два прежних val-снимка, prompts и настройки. Два отдельных процесса, по одной загрузке на режим; всего 32 ответа. Pinned upstream builder, без нашего адаптера/runner.

Сначала FP16, после выхода его процесса — NF4. Greedy, seed=42, llava_v1, 96 токенов. В NF4 CLIP/projector/lm_head остаются FP16. В полной FP16 сверяются все checkpoint tensors; в NF4 — visual/projector и пять неквантованных LM tensors. Входы одинаковы. Offload, другая точность и fallback запрещены.

**Не T4/16 GB:** gate требует минимум 32 GiB свободной VRAM (или размер checkpoint + 6 GiB, если больше). Это консервативный допуск, не гарантия отсутствия OOM. FP16 не ground truth; Torch/Transformers/LLaVA и merged-vision overrides общие. Выполненная v11 сохранена с выводами отдельно. Реальный GPU-контроль v12 здесь ещё не выполнен.


## Setup

Загрузите `spacellava_precision_v12.zip` в `MyDrive/planetary_benchmark/`. Там же нужен прежний `spacellava_evidence_v10_evidence_55164967_results.zip` (85 KB): только для фиксации входов и checkpoint identity. Старые ответы не заменяют новый NF4-прогон. При смене GPU оба режима выполняются заново на ней.

Выберите более ёмкую GPU, например класс 40 GB; фактический допуск определяет preflight. RUN_PILOT и DOWNLOAD_WEIGHTS по умолчанию False — включайте явно. После смены VM нужно восстановить окружение и локальные веса. Старые результаты сохраняются, resume запрещён. Не запускайте другие GPU-задачи одновременно.


In [ ]:
from pathlib import Path
import hashlib, json, os, signal, stat, subprocess, sys, uuid, zipfile, shutil
from google.colab import drive
drive.mount('/content/drive')
PERSIST = Path('/content/drive/MyDrive/planetary_benchmark')
BUNDLE_ZIP = PERSIST / 'spacellava_precision_v12.zip'
BASELINE_ZIP = PERSIST / 'spacellava_evidence_v10_evidence_55164967_results.zip'
WORKSPACE = Path('/content/planetary_precision_v12')
ENV_DIR = Path('/content/venv_spacellava')
PYTHON_VERSION = '3.12.15'
TORCH_INDEX = 'https://download.pytorch.org/whl/cu128'
RUN_ID = 'spacellava_precision_v12'
SESSION_ID = uuid.uuid4().hex
DOWNLOAD_WEIGHTS = False
WEIGHTS_SOURCE = None
REUSE_ENV = True
RUN_PILOT = False  # Explicitly enable FP16 then NF4, 16 requests each on the same GPU.
RESUME = False  # This diagnostic intentionally does not support resume; use a fresh SESSION_ID.
PERSIST.mkdir(parents=True, exist_ok=True)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.run(['nvidia-smi'], check=True)
assert BUNDLE_ZIP.is_file(), 'Upload spacellava_precision_v12.zip to the configured Drive directory.'
assert BASELINE_ZIP.is_file(), 'Provide the exact v10 result ZIP as BASELINE_ZIP; no new dataset is needed.'
assert hashlib.sha256(BASELINE_ZIP.read_bytes()).hexdigest() == '7ec827c442e2d556fe25b7f3092f3cffc94188fc8de5334402f6e3bef00d403e', 'Wrong or changed v10 result ZIP.'
print('Free local GiB:', round(shutil.disk_usage('/content').free/1024**3,1))


### 1. Проверить и распаковать переносимый пакет
Используй только пакет из своего проекта: в нём исполняемый код. Проверяются пути и SHA-256 всех файлов. Повторная ячейка не заменяет изменённые файлы. При новом пакете выбери новый workspace/run, не редактируй старые результаты.

In [ ]:
WORKSPACE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_ZIP) as archive:
    info = archive.infolist()
    names = [item.filename for item in info]
    assert len(names) == len(set(names)), 'Duplicate archive entry'
    record = json.loads(archive.read('colab_bundle.json'))
    claimed = record.pop('bundle_sha256')
    digest = hashlib.sha256(json.dumps(record, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()
    assert record['schema'] == 'planetary-colab-v1' and claimed == digest, 'Metadata changed'
    assert set(names) == set(record['files_sha256']) | {'colab_bundle.json'}, 'Unexpected archive entry'
    for item in info:
        relative = Path(item.filename)
        assert not relative.is_absolute() and '..' not in relative.parts
        assert chr(92) not in item.filename and ':' not in item.filename
        assert not any(ord(c) < 32 for c in item.filename)
        assert not set(p.casefold() for p in relative.parts) & {'data_rover', '.env', '.git', 'weights', 'external'}
        assert not stat.S_ISLNK(item.external_attr >> 16) and not item.flag_bits & 1
        target = (WORKSPACE / relative).resolve()
        assert target.is_relative_to(WORKSPACE.resolve())
        data = archive.read(item)
        expected = record['files_sha256'].get(item.filename, hashlib.sha256(data).hexdigest())
        assert hashlib.sha256(data).hexdigest() == expected
        if target.exists():
            assert target.read_bytes() == data, f'Existing file differs: {relative}'
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
print({key: record[key] for key in ['collection_images', 'selected_images', 'requests', 'excluded', 'counts']})
print(record['limitation'])
assert record['selected_images'] == 100 and record['requests'] == 400
assert record['protocol']['question_type'] == 'candidate_Q1_Q4_clean_unscored'
assert record['protocol']['conditions'] == ['clean']
assert record['protocol']['collection'] == 'orbital_300_v4'
print('Frozen pool verified. Two val images, sixteen Q1-Q4 requests PER precision arm; 32 responses total, no benchmark scoring.')


### 2. Изолированное окружение SpaceLLaVA
Ячейка скачивает Python/dependencies, но **не модель**. Kernel Colab не меняется: все команды модели запускаются отдельным Python 3.12.15. Это исключает конфликт stock Transformers/Tokenizers в Colab и не требует Rust. CUDA wheels — кандидат; следующий шаг должен проверить драйвер/GPU.
Не ставь upstream LLaVA через `pip install -e external/LLaVA`: там устаревшие Torch pins. IBM позже получит другое окружение.

In [ ]:
PYTHON = str(ENV_DIR / 'bin/python')
reuse = REUSE_ENV and Path(PYTHON).is_file()
if reuse:
    subprocess.run([PYTHON, '-c', "import platform, torch, transformers, bitsandbytes; assert platform.python_version() == '3.12.15'; assert torch.__version__.split('+')[0] == '2.8.0'; assert transformers.__version__ == '4.37.2'; assert bitsandbytes.__version__ == '0.48.2'"], check=True)
    print('Reusing pinned model environment; no dependency upgrade.')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv>=0.8,<1'], check=True)
    if not Path(PYTHON).exists():
        subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', PYTHON_VERSION, '--seed', str(ENV_DIR)], check=True)
    subprocess.run([PYTHON, '-m', 'pip', 'install', 'torch==2.8.0', 'torchvision==0.23.0', '--index-url', TORCH_INDEX], check=True)
    subprocess.run([PYTHON, '-m', 'pip', 'install', '--only-binary=tokenizers', '-r', str(WORKSPACE/'requirements/spacellava.txt')], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', str(WORKSPACE)], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)

def step(*arguments):
    command = [PYTHON, '-u', '-m', 'planetary_vlm.colab', *map(str, arguments), '--workspace', str(WORKSPACE)]
    logs = PERSIST / 'logs'
    logs.mkdir(exist_ok=True)
    import re
    assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]{0,90}', RUN_ID)
    log_path = logs / f'{RUN_ID}_{SESSION_ID[:8]}_{arguments[0]}.log'
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
    muted_meta = 0
    try:
        with log_path.open('a', encoding='utf-8') as log:
            for line in process.stdout:
                log.write(line); log.flush()
                if 'copying from a non-meta parameter' in line:
                    muted_meta += 1
                    if muted_meta > 2: continue
                if line.strip() == 'warnings.warn(': continue
                print(line, end='', flush=True)
        print('Full log:', log_path, '| repeated meta warnings:', muted_meta)
        code = process.wait()
        if code: raise subprocess.CalledProcessError(code, command)
    except BaseException:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait(timeout=10)
        raise
    finally:
        process.stdout.close()

step('verify')
step('environment')
step('precision-preflight')  # Stop on T4/16 GB BEFORE downloading weights.


## Steps
### 3. Локальные веса и pinned LLaVA
Для первой загрузки включи `DOWNLOAD_WEIGHTS=True` в параметрах и повтори эту ячейку. Веса публичные; helper не скачивает train-датасеты. Требуются не менее 35 GiB свободного локального диска; реальный размер пакетов/весов может меняться.
Если веса уже есть на Drive, задай `WEIGHTS_SOURCE` на папку с `SpaceLLaVA/` и `clip-vit-large-patch14-336/`. Копирование только на локальный диск, без изменения кеша. Git repo подключается по точному commit. HF_TOKEN, если нужен, передай через Colab Secrets/environment, не записывай в notebook/логи. Исходный SSL не отключать.

In [ ]:
for previous in (Path('/content/planetary_reference_v11'), Path('/content/planetary_evidence_v10'), Path('/content/planetary_fresh_v9'), Path('/content/planetary_spatial_v8'), Path('/content/planetary_language_v7'), Path('/content/planetary_q1_q4_v6_diagnostics'), Path('/content/planetary_q1_q4_v5'), Path('/content/planetary_colab')):
    for name in ('weights', 'external'):
        source, target = previous/name, WORKSPACE/name
        if source.is_dir() and not target.exists():
            target.symlink_to(source, target_is_directory=True)
            print('Reusing without copying:', target)


In [ ]:
if WEIGHTS_SOURCE is not None:
    cache = Path(WEIGHTS_SOURCE)
    for name in ['SpaceLLaVA', 'clip-vit-large-patch14-336']:
        destination = WORKSPACE / 'weights' / name
        if not destination.exists():
            shutil.copytree(cache / name, destination)
arguments = ['assets', '--clone-runtime']  # Small pinned code clone, even when using existing weights.
if DOWNLOAD_WEIGHTS: arguments.append('--download')
step(*arguments)
step('precision-preflight')  # Recheck with the actual checkpoint byte size.


### Проверить прежние изображения

Только два ранее использованных val-снимка. Moon_seg_012 — прежний контроль без учитываемых Q1-кратеров, а не утверждение о полном отсутствии любых округлых форм. Подписи предпросмотра и эталоны модели не передаются.


In [ ]:
pool = [json.loads(line) for line in (WORKSPACE/'data_orbital/colab_inputs/requests.jsonl').read_text().splitlines()]
origins = {row['request_id']: row for row in map(json.loads, (WORKSPACE/'data_orbital/colab_inputs/provenance.jsonl').read_text().splitlines())}
sample_ids = ('moon_seg_002', 'moon_seg_012')
from PIL import Image, ImageDraw
from IPython.display import display
preview = Image.new('RGB', (2*256, 282), 'white')
draw = ImageDraw.Draw(preview)
for index, sid in enumerate(sample_ids):
    selected = [r for r in pool if r['request_id'].startswith(sid+'__') and r['request_id'].endswith('__clean')]
    assert len(selected) == 4 and len({tuple(r['image_paths']) for r in selected}) == 1
    assert all(origins[r['request_id']]['source_split'] == 'val' for r in selected)
    path = WORKSPACE/selected[0]['image_paths'][0]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == origins[selected[0]['request_id']]['image_sha256']
    with Image.open(path) as image:
        preview.paste(image.convert('RGB').resize((256,256)), (index*256,26))
    draw.text((index*256+4,5), sid+' | val', fill='black')
display(preview)
print('Preview only: labels and this contact sheet are NOT passed to the model.')


## Pilot

В каждой ветке те же 16 запросов без смены порядка/формулировок. Два worker-процесса строго последовательно: FP16, затем NF4. Каждый независимо загружает checkpoint и создаёт новую conversation на запрос. Прежние ответы и ответы другой ветки не передаются worker-у.

Перед загрузками проверяются свободная память и UUID GPU. Core-среда не должна меняться между ветками. Checkpoint-файлы и входы одинаковы. Полная FP16-ветка сверяет все checkpoint tensors. Две neutral-forward пробы записываются в каждой ветке. Различие визуальных hash удерживает matched-control статус даже при завершении обоих режимов.

Результат — сравнение точности представления весов, не accuracy. Отличие текста не означает, что FP16 ответил правильно; совпадение не исключает влияния NF4 на других входах. Порядок веток фиксирован, не counterbalanced. Экспорт использует PRECISION_RUN; для нового запуска нужен новый SESSION_ID.


In [ ]:
if RUN_PILOT:
    pilot_name = RUN_ID+'_precision_'+SESSION_ID[:8]
    assert not (WORKSPACE/'outputs/SpaceLLaVA'/pilot_name).exists() and not (PERSIST/'runs/SpaceLLaVA'/pilot_name).exists(), 'Run already exists: set a new SESSION_ID before launching, keep old results.'
    PRECISION_RUN = dict(name=pilot_name, persist=str(PERSIST), run_id=RUN_ID, session_id=SESSION_ID)
    try:
        step('precision-pilot', '--persist', PERSIST, '--run-id', RUN_ID, '--session-id', SESSION_ID, '--baseline', BASELINE_ZIP)
    finally:
        result_dir = WORKSPACE/'outputs/SpaceLLaVA'/PRECISION_RUN['name']
        report_path = result_dir/'precision_comparison.json'
        if report_path.is_file():
            report = json.loads(report_path.read_text())
            print('Completed per arm:', report['completed_per_arm'], '| errors:', report['errors_per_arm'])
            print('Matched control:', report['matched_control'], '| same answers:', report['same_raw_responses'], '| different:', report['different_raw_responses'])
            for pair in report['comparisons']:
                print(pair['request_id'], '| same:', pair['same_raw_response'])
                print(' FP16:', pair['fp16']['raw_response'] if pair['fp16'] else 'NOT RUN')
                print(' NF4:', pair['nf4']['raw_response'] if pair['nf4'] else 'NOT RUN')
            print('NOT accuracy; benchmark admitted:', report['benchmark_admitted'])
else:
    print('Pilot disabled. Set RUN_PILOT=True explicitly: two sequential loads, sixteen requests each, same GPU.')


## Checks and export

ZIP сохраняет два PNG, вложенные ветки fp16/nf4 с jobs без эталонов, predictions/loading/probes/traces/logs, общий execution/comparison и console log. Имя берётся из PRECISION_RUN, не из изменённого SESSION_ID.

При OOM, неполной ветке или смене условий — остановка без fallback; после ошибки первой ветки вторая не запускается. Экспортируйте частичный результат. Если gate сработал до создания папки, пришлите console log из Drive/logs.


In [ ]:
assert 'PRECISION_RUN' in globals(), 'Launch the precision pilot first; do not invent or change a run ID for export.'
pilot_name = PRECISION_RUN['name']
run_persist = Path(PRECISION_RUN['persist'])
result_dir = run_persist/'runs/SpaceLLaVA'/pilot_name
assert result_dir.is_dir(), 'Send the console log if the pilot failed before result creation.'
export_path = run_persist/(pilot_name+'_results.zip')
with zipfile.ZipFile(export_path, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(result_dir.rglob('*')):
        assert not path.is_symlink(), 'Do not archive symlinks'
        if path.is_file():
            assert path.resolve().is_relative_to(result_dir.resolve())
            archive.write(path, arcname=pilot_name+'/'+path.relative_to(result_dir).as_posix())
    console = run_persist/'logs'/(PRECISION_RUN['run_id']+'_'+PRECISION_RUN['session_id'][:8]+'_precision-pilot.log')
    if console.is_file(): archive.write(console, arcname=pilot_name+'/console.log')
print('Send this ZIP:', export_path)
from google.colab import files
files.download(str(export_path))


## Next steps

Пришлите итоговый ZIP. Сравним FP16/NF4 на одной GPU: UUID, среда, checkpoint/input hashes, visual features, токены и текст. Никаких дополнительных prompts или sampling в этом контроле.

Реальный GPU-контроль v12 здесь не выполнен. Нужны Colab, достаточно ёмкая GPU, pinned runtime и локальный checkpoint. CPU-тесты не подтверждают VRAM/CUDA совместимость. Общие Torch/Transformers/LLaVA и merged-vision overrides остаются ограничением даже при совпадении результатов.
